In [112]:
import pandas as pd
from unidecode import unidecode

In [ ]:
df = pd.read_excel(r"data\Taco-4a-Edicao.xlsx")

In [114]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 597 entries, 0 to 596
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Número                 597 non-null    int64 
 1   Grupo                  597 non-null    str   
 2   Descrição do Alimento  597 non-null    str   
 3   Energia(kcal)          595 non-null    object
 4   Energia(kJ)            595 non-null    object
 5   Proteína(g)            586 non-null    object
 6   Lipídeos(g)            594 non-null    object
 7   Colesterol(mg)         267 non-null    object
 8   Carboidrato(g)         586 non-null    object
 9   Fibra Alimentar(g)     370 non-null    object
 10  Cálcio(mg)             586 non-null    object
 11  Magnésio(mg)           586 non-null    object
 12  Manganês(mg)           586 non-null    object
 13  Fósforo(mg)            586 non-null    object
 14  Ferro(mg)              586 non-null    object
 15  Sódio(mg)              589 non-nul

In [115]:
df.head()

,Número,Grupo,Descrição do Alimento,Energia(kcal),Energia(kJ),Proteína(g),Lipídeos(g),Colesterol(mg),Carboidrato(g),Fibra Alimentar(g),Cálcio(mg),Magnésio(mg),Manganês(mg),Fósforo(mg),Ferro(mg),Sódio(mg),Potássio(mg),Cobre(mg),Zinco(mg),VitaminaC(mg)
0,1,Cereais e derivados,"Arroz, integral, cozido",123.534893,516.86999,2.58825,1.000333,NaN,25.80975,2.749333,5.204,58.702,0.627333,105.853,0.262,1.244667,75.151667,0.020333,0.682667,NaN
1,2,Cereais e derivados,"Arroz, integral, cru",359.678002,1504.892761,7.323286,1.864833,NaN,77.450714,4.819167,7.818,109.71,2.993333,250.865,0.948333,1.645667,173.34,0.074833,1.395167,NaN
2,3,Cereais e derivados,"Arroz, tipo 1, cozido",128.258486,536.633504,2.520817,0.227,NaN,28.05985,1.561,3.544333,2.253333,0.296667,17.945,0.076667,1.200667,14.673667,0.015,0.491333,NaN
3,4,Cereais e derivados,"Arroz, tipo 1, cru",357.789273,1496.990319,7.15854,0.335,NaN,78.759543,1.639167,4.414333,30.383667,1.032333,104.207583,0.677747,1.019167,62.499417,0.114833,1.224833,NaN
4,5,Cereais e derivados,"Arroz, tipo 2, cozido",130.119648,544.420609,2.568417,0.361667,NaN,28.192583,1.069667,3.333667,6.053333,0.371,21.521333,0.050333,1.959667,20.203333,0.041,0.549333,NaN


TRATAMENTO

In [116]:
df_clean = df[["Número","Grupo","Descrição do Alimento","Energia(kcal)", 'Energia(kJ)', "Proteína(g)", "Carboidrato(g)", "Lipídeos(g)",'Fibra Alimentar(g)']]

In [117]:
df_clean = df_clean.rename(columns={'Número': 'id'})
df_clean.columns = [unidecode(col).lower() for col in df_clean.columns]

In [118]:
# aqui eu tentei dar uma comprimida nos tipos de variedade, eu quebro a descrição do alimento assim: 
# exemplo: arroz, integral, cozido → → → base = arroz | tipo = integral
# mantém 1 linha para cada combinação de (base + tipo) funcionou mais ou menos rsrs
def delete_rep(desc):
    partes = [p.strip().lower() for p in desc.split(",")]
    
    base = partes[0] if len(partes) > 0 else None
    tipo = partes[1] if len(partes) > 1 else None
    
    return pd.Series([base, tipo])

df_clean[["base", "tipo"]] = df_clean["descricao do alimento"].apply(delete_rep)

df_clean = df_clean.drop_duplicates(subset=["base", "tipo"])
df_clean = df_clean.drop(columns=["base", "tipo"])

In [120]:
colunas = [
    "energia(kcal)", "energia(kj)", "proteina(g)",
    "carboidrato(g)", "lipideos(g)", "fibra alimentar(g)"]

df_clean[colunas] = df_clean[colunas].apply(pd.to_numeric, errors="coerce")

df_clean = df_clean.round(1)

In [121]:
sorted(df_clean["grupo"].unique())

['Alimentos preparados',
 'Bebidas (alcoólicas e não alcoólicas)',
 'Carnes e derivados',
 'Cereais e derivados',
 'Frutas e derivados',
 'Gorduras e óleos',
 'Leguminosas e derivados',
 'Leite e derivados',
 'Miscelâneas',
 'Outros alimentos industrializados',
 'Ovos e derivados',
 'Pescados e frutos do mar',
 'Produtos açucarados',
 'Verduras, hortaliças e derivados']

In [122]:
df_clean = df_clean[~df_clean["grupo"].isin([
    "Alimentos preparados",
    "Bebidas (alcoólicas e não alcoólicas)",
    "Miscelâneas"
])]

In [123]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 376 entries, 0 to 596
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     376 non-null    int64  
 1   grupo                  376 non-null    str    
 2   descricao do alimento  376 non-null    str    
 3   energia(kcal)          374 non-null    float64
 4   energia(kj)            374 non-null    float64
 5   proteina(g)            363 non-null    float64
 6   carboidrato(g)         366 non-null    float64
 7   lipideos(g)            350 non-null    float64
 8   fibra alimentar(g)     272 non-null    float64
dtypes: float64(6), int64(1), str(2)
memory usage: 29.4 KB


In [124]:
df_clean.head(50)

,id,grupo,descricao do alimento,energia(kcal),energia(kj),proteina(g),carboidrato(g),lipideos(g),fibra alimentar(g)
0,1,Cereais e derivados,"Arroz, integral, cozido",123.5,516.9,2.6,25.8,1.0,2.7
2,3,Cereais e derivados,"Arroz, tipo 1, cozido",128.3,536.6,2.5,28.1,0.2,1.6
4,5,Cereais e derivados,"Arroz, tipo 2, cozido",130.1,544.4,2.6,28.2,0.4,1.1
6,7,Cereais e derivados,"Aveia, flocos, crua",393.8,1647.8,13.9,66.6,8.5,9.1
7,8,Cereais e derivados,"Biscoito, doce, maisena",442.8,1852.8,8.1,75.2,12.0,2.1
12,13,Cereais e derivados,"Biscoito, salgado, cream cracker",431.7,1806.4,10.1,68.7,14.4,2.5
13,14,Cereais e derivados,"Bolo, mistura para",418.6,1751.6,6.2,84.7,6.1,1.7
14,15,Cereais e derivados,"Bolo, pronto, aipim",323.9,1355.0,4.4,47.9,12.7,0.7
18,19,Cereais e derivados,"Canjica, branca, crua",357.6,1496.2,7.2,78.1,1.0,5.5
19,20,Cereais e derivados,"Canjica, com leite integral",112.5,470.5,2.4,23.6,1.2,1.2


In [ ]:
df_clean.to_excel("taco_reduzido.xlsx", index=False)

# ------------------------------------------------------------
# Nota:
# Foram realizados outros processos de limpeza e redução
# do dataset de forma manual, fora deste script.
# ------------------------------------------------------------